# Step 1 – BloodMNIST: blood cell type classification

**Capstone: Event-driven ML pipeline on Azure (ECE Paris – Distributed Systems and AI)**

Task: classify a microscope image of a blood cell into one of **8 types**.
Dataset: BloodMNIST (MedMNIST v2), 28×28 RGB, CC BY 4.0, official train/val/test split. *Not for clinical use.*

What this notebook does:
1. Downloads the dataset (`medmnist`, `download=True`)
2. Two naive / classic baselines (most-frequent class, PCA + logistic regression)
3. Small CNN trained from scratch
4. ResNet-18 pretrained on ImageNet, fine-tuned (transfer learning)
5. Metrics on the test set: **accuracy** and **macro-F1** + confusion matrix
6. Exports the best model as **`model-1.0.0.onnx`** (preprocessing + softmax included) and checks CPU latency < 1 s
7. Writes `metrics.json`, `labels.json`, `input_schema.json`, `requirements.txt`, demo images → `model_artifacts.zip`

**In Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (≈ 20–30 min).

In [ ]:
!pip install -q medmnist onnx onnxruntime onnxscript

## 1. Setup and reproducibility

In [ ]:
import os, json, time, random, platform, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
MODEL_VERSION = "1.0.0"
OUT = "artifacts"
os.makedirs(OUT, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

## 2. Download and explore BloodMNIST

In [ ]:
import medmnist
from medmnist import BloodMNIST, INFO

info = INFO["bloodmnist"]
CLASS_NAMES = [info["label"][str(i)] for i in range(len(info["label"]))]
print("Licence:", info.get("license"))
print("Classes:", CLASS_NAMES)

splits = {s: BloodMNIST(split=s, download=True, size=28) for s in ["train", "val", "test"]}
X = {s: d.imgs for s, d in splits.items()}                 # uint8, (N, 28, 28, 3)
y = {s: d.labels.reshape(-1).astype(np.int64) for s, d in splits.items()}
for s in X: print(s, X[s].shape, y[s].shape)

npz = os.path.expanduser("~/.medmnist/bloodmnist.npz")
if os.path.exists(npz): print(f"File size: {os.path.getsize(npz)/1e6:.1f} MB (limit 50 MB)")

In [ ]:
# Class balance
counts = np.bincount(y["train"], minlength=len(CLASS_NAMES))
plt.figure(figsize=(9, 3)); plt.bar(CLASS_NAMES, counts); plt.xticks(rotation=30, ha="right")
plt.title("Training images per class"); plt.tight_layout(); plt.savefig(f"{OUT}/class_balance.png"); plt.show()

# Sample grid
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for c, ax_col in enumerate(axes.T):
    idx = np.where(y["train"] == c)[0][:2]
    for ax, i in zip(ax_col, idx):
        ax.imshow(X["train"][i]); ax.axis("off")
    ax_col[0].set_title(CLASS_NAMES[c][:14], fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/samples.png"); plt.show()

## 3. Evaluation helper

In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    results[name] = {"accuracy": round(float(acc), 4), "macro_f1": round(float(f1), 4)}
    print(f"{name:28s} accuracy={acc:.4f}  macro-F1={f1:.4f}")
    return acc, f1

## 4. Baselines
*Baseline 1* always predicts the most frequent class. *Baseline 2* is a classic ML model on raw pixels (PCA 100 components + logistic regression).

In [ ]:
flat = {s: X[s].reshape(len(X[s]), -1).astype(np.float32) / 255.0 for s in X}

dummy = DummyClassifier(strategy="most_frequent").fit(flat["train"], y["train"])
evaluate("Baseline: most frequent", y["test"], dummy.predict(flat["test"]))

t0 = time.time()
pca_lr = make_pipeline(StandardScaler(), PCA(n_components=100, random_state=SEED),
                       LogisticRegression(max_iter=2000))
pca_lr.fit(flat["train"], y["train"])
evaluate("Baseline: PCA + LogReg", y["test"], pca_lr.predict(flat["test"]))
print(f"trained in {time.time()-t0:.1f}s")

## 5. PyTorch data loaders
Images are fed as float tensors in [0, 1], shape (N, 3, 28, 28). Normalisation and resizing happen **inside the model**, so the exported ONNX file takes raw pixels / 255 – the API stays simple.

In [ ]:
def to_tensor(a):  # uint8 NHWC -> float NCHW in [0,1]
    return torch.from_numpy(a).permute(0, 3, 1, 2).float() / 255.0

T = {s: to_tensor(X[s]) for s in X}
Y = {s: torch.from_numpy(y[s]) for s in y}

g = torch.Generator().manual_seed(SEED)
BATCH = 128
loaders = {
    "train": DataLoader(TensorDataset(T["train"], Y["train"]), batch_size=BATCH, shuffle=True, generator=g),
    "val":   DataLoader(TensorDataset(T["val"], Y["val"]), batch_size=256),
    "test":  DataLoader(TensorDataset(T["test"], Y["test"]), batch_size=256),
}

def augment(x):
    # random horizontal + vertical flips and 90° rotations (cells have no fixed orientation)
    if random.random() < 0.5: x = x.flip(3)
    if random.random() < 0.5: x = x.flip(2)
    k = random.randint(0, 3)
    return torch.rot90(x, k, dims=(2, 3)) if k else x

## 6. Models
Both models include preprocessing (resize + ImageNet normalisation) and the export wrapper adds softmax.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class Preprocess(nn.Module):
    def __init__(self, size=None):
        super().__init__(); self.size = size
        self.register_buffer("mean", MEAN.clone()); self.register_buffer("std", STD.clone())
    def forward(self, x):
        if self.size: x = F.interpolate(x, size=(self.size, self.size), mode="bilinear", align_corners=False)
        return (x - self.mean) / self.std

class SmallCNN(nn.Module):
    """3 conv blocks, ~0.3 M parameters, trained from scratch."""
    def __init__(self, n_classes=8):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                              nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.pre = Preprocess()
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))   # 28 -> 14 -> 7 -> 3
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x): return self.head(self.features(self.pre(x)))

class ResNet18FT(nn.Module):
    """ResNet-18 pretrained on ImageNet; input upscaled 28 -> 64 px; new 8-class head."""
    def __init__(self, n_classes=8, pretrained=True):
        super().__init__()
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.pre = Preprocess(size=64)
        self.net = torchvision.models.resnet18(weights=w)
        self.net.fc = nn.Linear(self.net.fc.in_features, n_classes)
    def forward(self, x): return self.net(self.pre(x))

## 7. Training loop (best epoch selected on the validation set by macro-F1)

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); preds = []
    for xb, _ in loader: preds.append(model(xb.to(device)).argmax(1).cpu())
    return torch.cat(preds).numpy()

def train(model, epochs, lr, name):
    set_seed()
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(loaders["train"]))
    best_f1, best_state, history = -1, None, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        for xb, yb in loaders["train"]:
            xb, yb = augment(xb).to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            total += loss.item() * len(xb)
        vf1 = f1_score(y["val"], predict(model, loaders["val"]), average="macro")
        history.append({"epoch": ep, "train_loss": total / len(T["train"]), "val_macro_f1": vf1})
        print(f"[{name}] epoch {ep:2d}  loss={total/len(T['train']):.4f}  val macro-F1={vf1:.4f}  ({time.time()-t0:.0f}s)")
        if vf1 > best_f1: best_f1, best_state = vf1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

In [ ]:
EPOCHS_CNN, EPOCHS_RESNET = 15, 10

cnn, hist_cnn = train(SmallCNN(len(CLASS_NAMES)), EPOCHS_CNN, 3e-3, "SmallCNN")
evaluate("SmallCNN (from scratch)", y["test"], predict(cnn, loaders["test"]))

In [ ]:
PRETRAINED = os.environ.get("NO_PRETRAINED") != "1"
resnet, hist_res = train(ResNet18FT(len(CLASS_NAMES), pretrained=PRETRAINED), EPOCHS_RESNET, 1e-3, "ResNet18")
evaluate("ResNet-18 (fine-tuned)", y["test"], predict(resnet, loaders["test"]))

## 8. Comparison and choice of the final model

In [ ]:
import pandas as pd
table = pd.DataFrame(results).T
print(table.to_markdown() if hasattr(table, "to_markdown") else table)

# Select on VALIDATION macro-F1 (the test set is only for reporting)
val_scores = {"SmallCNN (from scratch)": max(h["val_macro_f1"] for h in hist_cnn),
              "ResNet-18 (fine-tuned)": max(h["val_macro_f1"] for h in hist_res)}
best_name = max(val_scores, key=val_scores.get)
best_model = {"SmallCNN (from scratch)": cnn, "ResNet-18 (fine-tuned)": resnet}[best_name]
print("\nSelected model:", best_name, "| val macro-F1:", round(val_scores[best_name], 4))

In [ ]:
y_pred = predict(best_model, loaders["test"])
print(classification_report(y["test"], y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y["test"], y_pred), display_labels=CLASS_NAMES).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix – {best_name} (test set)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", dpi=120); plt.show()

## 9. Export to ONNX (preprocessing + softmax inside the file)
Input `image`: float32, shape `(batch, 3, 28, 28)`, values in [0, 1]. Output `probabilities`: float32 `(batch, 8)`.

In [ ]:
class ExportWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), dim=1)

export_model = ExportWrapper(best_model).cpu().eval()
onnx_path = f"{OUT}/model-{MODEL_VERSION}.onnx"
dummy_in = torch.rand(1, 3, 28, 28)
torch.onnx.export(export_model, dummy_in, onnx_path, input_names=["image"], output_names=["probabilities"],
                  dynamic_axes={"image": {0: "batch"}, "probabilities": {0: "batch"}}, opset_version=17, dynamo=False)
print(f"Saved {onnx_path}  ({os.path.getsize(onnx_path)/1e6:.1f} MB, limit 100 MB)")

## 10. Check ONNX = PyTorch, and CPU latency

In [ ]:
import onnxruntime as ort
so = ort.SessionOptions(); so.intra_op_num_threads = 1          # pessimistic: 1 CPU thread (Container Apps = 0.25 vCPU)
sess = ort.InferenceSession(onnx_path, so, providers=["CPUExecutionProvider"])

sample = T["test"][:64].numpy()
p_onnx = sess.run(None, {"image": sample})[0]
with torch.no_grad(): p_torch = export_model(torch.from_numpy(sample)).numpy()
print("max |ONNX - PyTorch| =", float(np.abs(p_onnx - p_torch).max()))
print("same predictions:", bool((p_onnx.argmax(1) == p_torch.argmax(1)).all()))

one = sample[:1]
for _ in range(5): sess.run(None, {"image": one})              # warm-up
lat = []
for i in range(100):
    t0 = time.perf_counter(); sess.run(None, {"image": sample[i % 64:i % 64 + 1]}); lat.append((time.perf_counter() - t0) * 1000)
lat_ms = {"mean": round(float(np.mean(lat)), 2), "p95": round(float(np.percentile(lat, 95)), 2)}
print(f"CPU latency per image (1 thread): mean {lat_ms['mean']} ms, p95 {lat_ms['p95']} ms  (limit 1000 ms)")

onnx_test_pred = np.concatenate([sess.run(None, {"image": T['test'][i:i+512].numpy()})[0].argmax(1)
                                 for i in range(0, len(T["test"]), 512)])
evaluate("Final ONNX model", y["test"], onnx_test_pred)

## 11. Save metadata, demo images and requirements

In [ ]:
from PIL import Image
import importlib.metadata as md_

metrics = {
    "model_version": MODEL_VERSION,
    "selected_model": best_name,
    "dataset": {"name": "BloodMNIST (MedMNIST v2)", "url": "https://medmnist.com/", "licence": info.get("license", "CC BY 4.0"),
                "image_size": "28x28 RGB", "n_train": int(len(y["train"])), "n_val": int(len(y["val"])), "n_test": int(len(y["test"]))},
    "seed": SEED,
    "test_metrics": results,
    "cpu_latency_ms_1_thread": lat_ms,
    "onnx_size_mb": round(os.path.getsize(onnx_path) / 1e6, 2),
    "trained_on": {"device": device, "torch": torch.__version__, "python": platform.python_version()},
}
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), indent=2)
json.dump({str(i): n for i, n in enumerate(CLASS_NAMES)}, open(f"{OUT}/labels.json", "w"), indent=2)

# Contract shared with the dispatcher (file validation) and the API (/predict)
schema = {
    "accepted_extensions": [".png", ".jpg", ".jpeg"],
    "max_file_size_bytes": 1_000_000,
    "image": {"mode": "RGB", "resize_to": [28, 28]},
    "onnx_input": {"name": "image", "dtype": "float32", "shape": ["batch", 3, 28, 28], "range": [0, 1]},
    "onnx_output": {"name": "probabilities", "dtype": "float32", "shape": ["batch", len(CLASS_NAMES)]},
}
json.dump(schema, open(f"{OUT}/input_schema.json", "w"), indent=2)

# 3 demo images per class (from the TEST set) for uploads during the video / defence
os.makedirs(f"{OUT}/demo_images", exist_ok=True)
for c, name in enumerate(CLASS_NAMES):
    for k, i in enumerate(np.where(y["test"] == c)[0][:3]):
        Image.fromarray(X["test"][i]).save(f"{OUT}/demo_images/{name.split(' ')[0]}_{k}.png")

# Pinned requirements = the exact versions used for this run
pkgs = ["torch", "torchvision", "medmnist", "numpy", "scikit-learn", "onnx", "onnxruntime", "onnxscript", "matplotlib", "pandas", "pillow"]
with open(f"{OUT}/requirements.txt", "w") as f:
    for p in pkgs:
        try: f.write(f"{p}=={md_.version(p).split('+')[0]}\n")
        except md_.PackageNotFoundError: pass
print(open(f"{OUT}/requirements.txt").read())
print(json.dumps(metrics, indent=2))

In [ ]:
zip_path = "model_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), OUT))
print("Created", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    pass

## What goes into the GitHub repo (`/model/`)
- this notebook (`bloodmnist_training.ipynb`)
- `requirements.txt`, `metrics.json`, `labels.json`, `input_schema.json`
- `model-1.0.0.onnx`: SmallCNN ≈ 1 MB, ResNet-18 ≈ 45 MB. Both fit GitHub's 100 MB file limit; you can also keep it in the `models/` blob container
- `confusion_matrix.png`, `class_balance.png` → `/docs/`
- **not** `bloodmnist.npz` (add `*.npz` to `.gitignore`; the notebook downloads it)

`input_schema.json` is the contract for teammates: the dispatcher uses it to accept/reject files, the API uses it to preprocess images.